# 🔧 对接大模型与工具调用（Function Calling + MCP）

---

## 课程导航

| 部分 | 内容 | 对应代码 |
|------|------|----------|
| **Part 1** | 工具调用闭环概述 | — |
| **Part 2** | 环境准备与项目结构 | `backend/` |
| **Part 3** | 从「硬编码」到「配置化 LLM 封装」 | `backend/llm/client.py` |
| **Part 4** | 从「if/else」到「工具注册表」 | `backend/tools/builtin_tools.py` |
| **Part 5** | 从「进程内函数」到「MCP 工具」 | `mcp_servers/web_search_server.py` + `backend/mcp_client/mcp_client.py` |
| **Part 6** | Function Calling 循环 | `backend/agents/single_agent.py` |
| **Part 7** | 把一切串起来（API 与前端） | `backend/main.py` + `backend/api/routes.py` + `frontend/` |
| **Part 8** | 课程总结 + 课后练习 | — |

---

> **面向学员**：本教程面向弱基础学员，每个知识点都有详细讲解。
> 建议按顺序逐 Cell 运行。这不是「背代码」，而是理解「代码为什么这么写」。

⚙️ Jupyter 异步运行说明

本 notebook 在 Jupyter 内核的自动事件循环中运行，异步调用直接写顶层 `await`，
不要用 `asyncio.run()`（会报 `RuntimeError`）。

- ✅ 正确：`await demo_chat()`
- ❌ 错误：`asyncio.run(demo_chat())`


---
## Part 1：工具调用闭环概述
---

### 1.1 大模型的「知识边界」

大模型（LLM）有三个天生短板：

- ❌ **不知道当前时间**（训练数据有时间截止点）
- ❌ **算不准数学**（如 `12345 * 6789`）
- ❌ **不会实时联网**（无法知道最新市场动态）

解决办法：**给模型加工具（Tools）**。

### 1.2 什么是 Function Calling？

**Function Calling** 让「大模型自主决定调用哪个函数、给出参数」：

> 用户说「现在几点了」→ 模型不编时间，而是说「我要调用 current_time」→ 程序执行函数 → 把结果给模型 → 模型组织语言回答。

### 1.3 什么是 MCP（Model Context Protocol）？

MCP 是 AI 领域的「USB 接口」标准：

- **Server 端**：工具/资源的提供方（如联网搜索服务）
- **Client 端**：消费工具的应用方（我们的后端）

只要 Server 符合 MCP 协议，任何 Client 都能直接使用，无需专属集成代码。

### 1.4 工具调用闭环总览

```text
用户输入（自然语言）
   │
   ▼
① 单智能体组装 messages（含对话历史）
   │
   ▼
② 调用大模型（携带 内置工具 Schema + MCP 工具 Schema）
   │
   ├── 模型返回 tool_calls？ ─── 否 ──→ ③ 直接返回最终回答
   │
   └── 是
       ▼
④ 按工具名路由执行：
   ├── 内置工具（current_time / calculator）→ 本地 Python 函数
   └── MCP 工具（web_search）→ MCP Client 按 Server 路由
       ├── stdio Server（内置 web_search，FastMCP）→ 子进程会话
       └── Streamable HTTP Server（自定义，如 12306）→ 远程 HTTP
       │
       ▼
⑤ 工具结果追加回 messages（role: tool）→ 回到 ②（最多 5 轮）
```

本节课就是把上图从上到下完整实现一遍。

In [1]:
# ============================================================
# Cell 1：环境准备 —— Python 版本 + 依赖检查
# ============================================================
import sys
import os
from pathlib import Path

print('=' * 55)
print('  环境检查')
print('=' * 55)
print(f'  Python 版本: {sys.version.split()[0]}')
print(f'  工作目录: {os.getcwd()}')

# 自动定位 backend/（不依赖当前工作目录）
# 教学要点：sys.path 是 Python 查找模块的搜索路径列表。找不到 mcp_client
# 报 ModuleNotFoundError，多半是这一步没执行，或工作目录不在 smartbrief/ 下。
# 这里先"向上"找含 backend/ 的父目录，找不到再"向下"找常见子目录。
def _find_root(start: Path) -> Path:
    p = start
    while True:
        if (p / 'backend').exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    for sub in sorted(start.iterdir()):
        if sub.is_dir() and (sub / 'backend').exists():
            return sub
    raise RuntimeError('未找到 backend/ 目录，请把工作目录切换到 smartbrief/')

root = _find_root(Path.cwd())
sys.path.insert(0, str(root / 'backend'))
sys.path.insert(0, str(root))
print(f'  项目根目录: {root}')
print('  [OK] 已将 backend 加入 sys.path')

# 核心依赖导入验证
for lib, name in [('fastapi', 'FastAPI'), ('openai', 'OpenAI SDK'),
                  ('mcp', 'MCP SDK'), ('dotenv', 'python-dotenv')]:
    try:
        __import__(lib)
        print(f'  [OK] {name}')
    except ImportError as exc:
        print(f'  [NO] {name} 导入失败: {exc}')

print('=' * 55)
print('  提示：请从 Cell 1 开始按顺序运行本 notebook')


  环境检查
  Python 版本: 3.10.20
  工作目录: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj02\smartbrief
  项目根目录: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj02\smartbrief
  [OK] 已将 backend 加入 sys.path
  [OK] FastAPI
  [OK] OpenAI SDK
  [OK] MCP SDK
  [OK] python-dotenv
  提示：请从 Cell 1 开始按顺序运行本 notebook


---
## Part 2：环境准备与项目结构
---


### 2.1 项目结构一览

```text
smartbrief/
├── backend/                     # 后端（FastAPI）
│   ├── main.py                  # 入口：lifespan 启动 MCP / 初始化工具注册表
│   ├── api/routes.py            # API 路由：/chat、/tools、/tools/mcp ...
│   ├── llm/client.py            # 大模型封装（OpenAI 兼容接口）← Part 3
│   ├── tools/builtin_tools.py   # 内置工具注册表 ← Part 4
│   ├── mcp_client/mcp_client.py # MCP 客户端（ClientSession）← Part 5
│   ├── agents/single_agent.py   # 单智能体 Function Calling 循环 ← Part 6
│   ├── .env                     # API Key 配置（不提交 git）
│   ├── tool_config.json         # 内置工具开关持久化
│   └── mcp_config.json          # MCP Server 配置持久化
├── mcp_servers/web_search_server.py  # MCP Server（FastMCP，stdio）← Part 5
└── frontend/                    # Vue3 前端 ← Part 7
```

### 2.2 密钥配置（API Key 不硬编码）

密钥放 `backend/.env`，代码用 `python-dotenv` 读取：

```text
OPENAI_BASE_URL=https://api.deepseek.com/v1
OPENAI_API_KEY=sk-xxxx
OPENAI_MODEL=deepseek-chat
```

任何 OpenAI 兼容平台（DeepSeek / Kimi / Ollama）都能用，只需改这三行。

In [2]:
# ============================================================
# Cell 2：查看 backend/.env.example —— 密钥配置模板
# ============================================================
from pathlib import Path

env_example = Path('backend/.env.example')
print('=' * 55)
print('  backend/.env.example 内容')
print('=' * 55)
print(env_example.read_text(encoding='utf-8'))

env_file = Path('backend/.env')
if env_file.exists():
    has_key = any(
        line.strip().startswith('OPENAI_API_KEY=') and len(line.split('=', 1)[1].strip()) > 8
        for line in env_file.read_text(encoding='utf-8').splitlines()
    )
    print(f'[提示] backend/.env 存在，OPENAI_API_KEY 已配置: {has_key}')
    print('       （未配置的话，Part 3 / Part 6 中调用大模型的演示会跳过）')
else:
    print('[提示] backend/.env 不存在，请参考 .env.example 创建')

  backend/.env.example 内容
# SmartBrief 后端配置示例：复制为 .env 并填入真实值
# 支持任意 OpenAI 兼容接口（DeepSeek / Kimi / Ollama 等），只需改 BASE_URL 与 MODEL

OPENAI_BASE_URL=https://api.openai.com/v1
OPENAI_API_KEY=sk-xxxx
OPENAI_MODEL=gpt-4o-mini

[提示] backend/.env 存在，OPENAI_API_KEY 已配置: True
       （未配置的话，Part 3 / Part 6 中调用大模型的演示会跳过）


---
## Part 3：从「硬编码」到「配置化 LLM 封装」
---

### 3.1 需求：先让模型能对话

写大模型应用，第一步就是调用 LLM API。新手写法：

In [3]:
# ============================================================
# Cell 3：v0 版本 —— 硬编码，能跑就行
# ============================================================
BASE_URL = 'https://api.deepseek.com/v1'
API_KEY = 'sk-xxxxxx'      # ❌ 密钥写死在代码里！
MODEL = 'deepseek-chat'

print('[NO] 痛点：')
print('   1. 密钥泄露风险：代码提交 git 就公开了')
print('   2. 换平台要改代码：切到 Kimi / Ollama 必须改三处')
print('   3. 每个文件都要复制一遍配置')
print()
print('[提示] 正确做法：配置与代码分离 —— 密钥放 .env，代码读环境变量')

[NO] 痛点：
   1. 密钥泄露风险：代码提交 git 就公开了
   2. 换平台要改代码：切到 Kimi / Ollama 必须改三处
   3. 每个文件都要复制一遍配置

[提示] 正确做法：配置与代码分离 —— 密钥放 .env，代码读环境变量


### 3.2 重构：配置化 + 懒加载单例

项目里 `backend/llm/client.py` 的解决方式：

- **配置化**：`base_url` / `api_key` / `model` 全部从 `.env` 读取，带默认值
- **懒加载单例**：`get_client()` 只在第一次调用时创建客户端
- **统一入口**：`chat(messages, tools)` 封装 Chat Completions，支持 `tools` 参数

In [4]:
# ============================================================
# Cell 4：看看项目里的最终代码 —— llm/client.py
# ============================================================
import inspect
from llm import client as llm_client

print('=' * 55)
print('  llm/client.py —— 源码')
print('=' * 55)
print(inspect.getsource(llm_client))

print('[提示] 三个设计要点：')
print('   1. load_dotenv() 让代码能读取 backend/.env')
print('   2. get_client() 懒加载：无 key 时报错提示，不静默失败')
print('   3. chat() 支持 tools / tool_choice —— 这是 Function Calling 的入口')

  llm/client.py —— 源码
"""大模型封装：OpenAI 兼容接口（第 22 节）。

设计动机（教学要点）：
- 整个项目所有"需要大模型"的地方（单智能体循环、内置 mock 技能、自定义技能）
  都只依赖本模块暴露的三个函数：get_client / get_model / chat。
  好处是换模型、换接口地址只改这一处，其余代码零改动。
- 配置从 backend/.env 读取（.env 不提交到 git），遵循「密钥不进代码」的安全约定：
  - OPENAI_BASE_URL：接口地址（默认 https://api.openai.com/v1，可指向 DeepSeek 等兼容服务）
  - OPENAI_API_KEY：API Key（必填）
  - OPENAI_MODEL：模型名（默认 gpt-4o-mini）
"""
import os

from dotenv import load_dotenv
from openai import AsyncOpenAI

# 从 backend/.env 加载配置（load_dotenv 会读取当前工作目录下的 .env；
# 后端统一在 backend/ 目录启动，因此能读到 backend/.env）
load_dotenv()

# 模块级全局缓存：AsyncOpenAI 客户端是重量级对象（内部维护连接池），
# 应该全局只创建一次（单例），而不是每次调用都 new 一个。
_client: AsyncOpenAI | None = None


def get_client() -> AsyncOpenAI:
    """获取（懒加载）AsyncOpenAI 客户端。

    教学要点——什么是"懒加载单例"：
    - 第一次调用时才真正创建对象（懒：lazy），之后一直复用同一个实例（单例：singleton）。
    - 好处：① 避免重复创建连接池的开销；② 启动阶段不联网，配置缺失时
      只在真正调用大模型的那一刻才报错，程序可以正常启动、正常走健康检查。
    """
    global _client
    if _client is None:
        base_url = os.getenv("OPENAI_BASE_URL

In [5]:
# ============================================================
# Cell 5：演示 —— 调用 chat() 与模型对话（需要 .env key + 网络）
# ============================================================
import asyncio
from llm.client import chat, get_model

async def demo_chat():
    try:
        resp = await chat([{'role': 'user', 'content': '你好，请用一句话介绍你自己'}])
        msg = resp.choices[0].message
        print('[OK] LLM 调用成功')
        print(f'  模型: {get_model()}')
        print(f'  回复: {msg.content}')
    except Exception as exc:
        print('[NO] LLM 调用失败（可能未配置 .env 或网络不可达）')
        print(f'  原因: {exc}')

await demo_chat()

[OK] LLM 调用成功
  模型: deepseek-v4-flash
  回复: 你好！我是DeepSeek，由深度求索公司创造的AI助手，随时乐意为你解答问题、提供帮助。


---
## Part 4：从「if/else」到「工具注册表」
---

### 4.1 需求升级：让模型能查时间、能算数

对话能跑了，但用户问「现在几点了」，模型只能瞎编。先看新手写法：

In [6]:
# ============================================================
# Cell 6：v1 版本 —— 手动 if/else 猜意图
# ============================================================
from datetime import datetime

# 假设我们要识别「查时间 / 算数」两个意图：
def handle_old_way(text):
    if '时间' in text or '几点' in text or '日期' in text:
        return f'现在是 {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
    if any(c in text for c in '+-*/'):
        return '计算功能（待实现）'
    return '我不知道'

print('[NO] 痛点：')
print('   1. 只能匹配「写死的关键词」——用户换个说法就失效')
print('   2. 意图识别规则越写越多，永远写不完')
print('   3. 计算逻辑还得自己解析表达式，又复杂又不安全')
print()
print('[思考] 与其让「程序猜意图」，不如让「模型猜意图」——模型读自然语言的能力远强于正则！')

[NO] 痛点：
   1. 只能匹配「写死的关键词」——用户换个说法就失效
   2. 意图识别规则越写越多，永远写不完
   3. 计算逻辑还得自己解析表达式，又复杂又不安全

[思考] 与其让「程序猜意图」，不如让「模型猜意图」——模型读自然语言的能力远强于正则！


### 4.2 关键思想：JSON Schema 是「模型与函数之间的契约」

Function Calling 的核心是 **JSON Schema**：

- 用 Schema 描述「函数叫什么、需要什么参数、参数是什么类型」
- 把 Schema 列表注入模型的 `tools` 参数
- 模型读用户的话，**自主决定**是否调用某个函数，并生成符合 Schema 的参数

```json
{
  "type": "function",
  "function": {
    "name": "calculator",
    "description": "执行四则运算",
    "parameters": {
      "type": "object",
      "properties": {"expression": {"type": "string"}},
      "required": ["expression"]
    }
  }
}
```

> 模型返回的不是结果，而是「我想调用 calculator，参数是 1 + 2 * 3」——执行权始终在我们手上。

In [7]:
# ============================================================
# Cell 7：v2 版本 —— 把函数「注册」起来（注册表模式雏形）
# ============================================================
from dataclasses import dataclass
from datetime import datetime

@dataclass
class SimpleTool:
    """一个工具的元信息：名字 + 描述 + 执行函数"""
    name: str
    description: str
    schema: dict
    handler: object

def current_time():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def calculator(expression: str):
    return f'（计算 {expression}，安全求值见后文）'

tools_registry = {
    'current_time': SimpleTool(
        name='current_time', description='获取当前时间',
        schema={'type': 'function', 'function': {'name': 'current_time',
                 'description': '获取当前时间',
                 'parameters': {'type': 'object', 'properties': {}, 'required': []}}},
        handler=current_time,
    ),
    'calculator': SimpleTool(
        name='calculator', description='执行四则运算',
        schema={'type': 'function', 'function': {'name': 'calculator',
                 'description': '四则运算',
                 'parameters': {'type': 'object',
                  'properties': {'expression': {'type': 'string'}},
                  'required': ['expression']}}},
        handler=calculator,
    ),
}

# 统一的执行入口：按名字路由
def execute_tool(name, arguments):
    tool = tools_registry.get(name)
    if tool is None:
        return {'error': f'未知工具: {name}'}
    if arguments:
        return tool.handler(**arguments)
    return tool.handler()

print('[模拟] 模型返回 tool_calls: [current_time]')
print('[OK] 执行结果:', execute_tool('current_time', {}))
print('[模拟] 模型返回 tool_calls: [calculator, expression=1+2*3]')
print('[OK] 执行结果:', execute_tool('calculator', {'expression': '1 + 2 * 3'}))

[模拟] 模型返回 tool_calls: [current_time]
[OK] 执行结果: 2026-09-01 00:03:33
[模拟] 模型返回 tool_calls: [calculator, expression=1+2*3]
[OK] 执行结果: （计算 1 + 2 * 3，安全求值见后文）


In [8]:
# ============================================================
# Cell 8：看看项目里的最终代码 —— builtin_tools.py 注册表
# ============================================================
from tools import builtin_tools

print('=' * 55)
print('  BuiltinTool 数据类')
print('=' * 55)
print(inspect.getsource(builtin_tools.BuiltinTool))

print('=' * 55)
print('  注册表核心方法')
print('=' * 55)
print(inspect.getsource(builtin_tools.init))
print(inspect.getsource(builtin_tools.list_enabled_schemas))

print('[提示] 相比 v2，项目版多了什么？')
print('   1. enabled 开关：关闭的工具不注入 LLM，但保留在注册表')
print('   2. 持久化：开关状态保存到 backend/tool_config.json')
print('   3. 分类元数据：metadata 里的 category 供前端分组展示')

  BuiltinTool 数据类
@dataclass
class BuiltinTool:
    """内置工具描述：名称 / 描述 / OpenAI Schema / 执行函数 / 开关状态。"""

    name: str
    description: str
    schema: dict
    handler: object  # Callable[..., Awaitable[str] | str]
    enabled: bool = True
    metadata: dict = field(default_factory=dict)  # 扩展信息（分类、参数说明等）

  注册表核心方法
def init() -> None:
    """初始化注册表：注册全部内置工具，并应用持久化的开关状态。

    教学要点：
    - 先 clear() 再注册：init 可能在应用重启时被多次调用（如 lifespan），
      先清空旧表避免残留上一次的重复记录。
    - 注册完毕后再从 tool_config.json 读取开关状态并覆盖 enabled，
      实现"开关配置跨重启保持"。
    """
    _TOOLS.clear()
    _register(
        BuiltinTool(
            name="current_time",
            description="获取服务器当前日期和时间",
            schema=current_time_schema(),
            handler=current_time,
            metadata={"category": "system", "params": "无参数"},
        )
    )
    _register(
        BuiltinTool(
            name="calculator",
            description="执行四则运算（加/减/乘/除/括号）",
            schema=calculator_schema(),
            handler=cal

In [9]:
# ============================================================
# Cell 9：演示 —— 初始化注册表 + 直接调用内置工具
# ============================================================
import asyncio
builtin_tools.init()

print('=' * 55)
print('  内置工具列表')
print('=' * 55)
for t in builtin_tools.list_tools():
    print(f'  • {t["name"]}（{t["description"]}）enabled={t["enabled"]}')

async def demo_tools():
    print()
    print('[OK] current_time 执行结果:', await builtin_tools.current_time())
    print('[OK] calculator 执行结果:', await builtin_tools.calculator('(1 + 2) * 3 / 2'))
    print('[OK] calculator 非法输入:', await builtin_tools.calculator('1 +'))

await demo_tools()

  内置工具列表
  • current_time（获取服务器当前日期和时间）enabled=True
  • calculator（执行四则运算（加/减/乘/除/括号））enabled=True

[OK] current_time 执行结果: 2026-09-01 00:03:33
[OK] calculator 执行结果: 4.5
[OK] calculator 非法输入: 计算错误：invalid syntax (<unknown>, line 1)


### 4.3 安全专题：为什么「绝不」用 eval 计算表达式？

`calculator` 收到的 `expression` 来自用户的自然语言。如果直接 `eval(expression)`：

- 用户输入 `1 + 1` → 结果 2 ✅
- 用户输入 `__import__('os').system('rm -rf /')` → **你的服务器被删了** 💥

`eval` 会执行任意 Python 代码，这是教科书级的严重漏洞。

In [10]:
# ============================================================
# Cell 10：演示 eval 的致命漏洞（只执行无害命令做演示）
# ============================================================
print('=' * 55)
print('  eval 漏洞演示')
print('=' * 55)

# 恶意表达式：通过 eval 读取服务器环境变量
malicious = "__import__('os').getcwd()"
print(f'  恶意输入: {malicious}')
print(f'  eval 结果: {eval(malicious)}')
print()
print('[NO] 可怕之处：eval 让用户输入变成了任意代码执行')
print('     换成 system(...) 就是直接删库删服务器')
print()
print('[方案] 项目做法：用 ast.parse 解析成语法树，再按「白名单」逐节点递归求值')
print('     白名单只放行 数字/加减乘除/括号/负号，其他一律拒绝')

  eval 漏洞演示
  恶意输入: __import__('os').getcwd()
  eval 结果: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj02\smartbrief

[NO] 可怕之处：eval 让用户输入变成了任意代码执行
     换成 system(...) 就是直接删库删服务器

[方案] 项目做法：用 ast.parse 解析成语法树，再按「白名单」逐节点递归求值
     白名单只放行 数字/加减乘除/括号/负号，其他一律拒绝


In [11]:
# ============================================================
# Cell 11：演示 —— 项目版 AST 安全求值
# ============================================================
import ast
from tools.builtin_tools import _safe_eval

print('=' * 55)
print('  安全求值：合法表达式')
print('=' * 55)
for expr in ['1 + 2 * 3', '(10 - 4) / 2', '2 ** 10', '7 % 3', '-5 + 8']:
    tree = ast.parse(expr, mode='eval')
    print(f'  {expr:>12} = {_safe_eval(tree)}')

print()
print('=' * 55)
print('  安全求值：恶意/越界表达式（应全部被拒绝）')
print('=' * 55)
attacks = [
    "__import__('os').getcwd()",
    "open('/etc/passwd').read()",
    "[1,2,3][0]",
    "'a' * 999",
]
for expr in attacks:
    try:
        tree = ast.parse(expr, mode='eval')
        _safe_eval(tree)
        print(f'  [NO] 竟然通过了: {expr}')
    except (ValueError, SyntaxError) as exc:
        print(f'  [OK] 已拒绝: {expr!r:45} -> {exc}')

print()
print('[提示] 原理三步走：')
print('   1. ast.parse 把字符串解析成语法树（不执行任何代码）')
print('   2. 递归遍历节点，只放行 Constant(数字) / BinOp(白名单运算) / UnaryOp(负号)')
print('   3. 出现函数调用 Call、属性访问 Attribute、下标 Subscript → 抛错拒绝')

  安全求值：合法表达式
     1 + 2 * 3 = 7
  (10 - 4) / 2 = 3.0
       2 ** 10 = 1024
         7 % 3 = 1
        -5 + 8 = 3

  安全求值：恶意/越界表达式（应全部被拒绝）
  [OK] 已拒绝: "__import__('os').getcwd()"                   -> 仅支持数字、四则运算与括号
  [OK] 已拒绝: "open('/etc/passwd').read()"                  -> 仅支持数字、四则运算与括号
  [OK] 已拒绝: '[1,2,3][0]'                                  -> 仅支持数字、四则运算与括号
  [OK] 已拒绝: "'a' * 999"                                   -> 仅支持数字、四则运算与括号

[提示] 原理三步走：
   1. ast.parse 把字符串解析成语法树（不执行任何代码）
   2. 递归遍历节点，只放行 Constant(数字) / BinOp(白名单运算) / UnaryOp(负号)
   3. 出现函数调用 Call、属性访问 Attribute、下标 Subscript → 抛错拒绝


In [12]:
# ============================================================
# Cell 12：演示 —— 注册表开关：关闭的工具不再注入 LLM
# ============================================================
print('=' * 55)
print('  工具开关演示')
print('=' * 55)

before = [s['function']['name'] for s in builtin_tools.list_enabled_schemas()]
print(f'  开启的工具: {before}')

# 关闭 calculator
builtin_tools.set_enabled('calculator', False)
after = [s['function']['name'] for s in builtin_tools.list_enabled_schemas()]
print(f'  关闭 calculator 后注入的 Schema: {after}')
print(f'  注册表中仍可查: get_tool("calculator").enabled = '
      f'{builtin_tools.get_tool("calculator").enabled}')

# 恢复开启（教学演示，不影响项目运行）
builtin_tools.set_enabled('calculator', True)
print(f'  [OK] 已恢复: '
      f'{[s["function"]["name"] for s in builtin_tools.list_enabled_schemas()]}')
print()
print('[注意] set_enabled 会写 backend/tool_config.json —— 这就是开关的持久化')

  工具开关演示
  开启的工具: ['current_time', 'calculator']
  关闭 calculator 后注入的 Schema: ['current_time']
  注册表中仍可查: get_tool("calculator").enabled = False
  [OK] 已恢复: ['current_time', 'calculator']

[注意] set_enabled 会写 backend/tool_config.json —— 这就是开关的持久化


---
## Part 5：从「进程内函数」到「MCP 工具」
---

### 5.1 需求：让模型能联网搜索

内置工具都是「我们自己写的 Python 函数」。但联网搜索这种能力，
每个应用都要自己爬搜索引擎、解析 HTML、处理反爬……太痛苦了。

**MCP 的答案**：把「联网搜索」做成一个标准的 MCP Server，任何应用都能当 Client 接入。

### 5.2 Server 端：FastMCP（`mcp_servers/web_search_server.py`）

FastMCP 让「写一个 MCP Server」简化到极致：**普通函数 + 一个装饰器**。

- `@mcp.tool()` 装饰后，**函数签名（参数名/类型/默认值/docstring）就是工具的 JSON Schema**
- FastMCP 自动处理 stdio 传输、initialize / tools/list / tools/call 全会话流程
- 由后端 MCPManager 以**子进程**方式启动，通过 stdin/stdout 通信

In [13]:
# ============================================================
# Cell 13：看 Server 端源码 + 直接调用搜索函数
# ============================================================
import mcp_servers.web_search_server as ws

print('=' * 55)
print('  web_search_server.py —— 工具定义')
print('=' * 55)
print(inspect.getsource(ws.web_search))
print(inspect.getsource(ws._mock_search))

print('=' * 55)
print('  直接调用 web_search 函数（真实搜索或 mock 降级）')
print('=' * 55)
import json
result = json.loads(ws.web_search('便携储能 2026 市场', 3))
print(f'  数据来源: {result["source"]}')
print(f'  查询词: {result["query"]}')
for i, item in enumerate(result['results'], 1):
    print(f'  {i}. {item["title"]}')
    print(f'     {item["snippet"][:40]}...')

  web_search_server.py —— 工具定义
@mcp.tool()
def web_search(query: str, num_results: int = 5) -> str:
    """联网搜索互联网，返回标题、摘要与链接列表。

    :param query: 搜索关键词
    :param num_results: 返回结果数量（默认 5，最大 10）

    教学要点：
    - 函数签名（参数名 + 类型 + 默认值 + docstring）就是工具的 JSON Schema 来源，
      FastMCP 会自动生成 `inputSchema` 并注入 LLM 的 tools 列表。
    - 返回值统一用 JSON 字符串（确保中文不转义），Client 端会尝试解析回 dict。
    """
    try:
        num_results = int(num_results)      # 兼容 LLM 有时把整数参数传成字符串
    except (ValueError, TypeError):
        num_results = 5                     # 非法值回退默认数量
    num_results = min(max(num_results, 1), 10)  # 限幅：最少 1 条、最多 10 条
    result = _web_search(query, num_results)
    return json.dumps(result, ensure_ascii=False)

def _mock_search(query: str, num_results: int = 5) -> dict:
    """降级：返回固定的模拟搜索结果（保证演示闭环）。"""
    results = [
        {
            "title": f"「{query}」模拟搜索结果 {i + 1}",
            "snippet": f"这是关于「{query}」的第 {i + 1} 条模拟摘要，"
                       f"用于课程演示 MCP 工具调用闭环。",
            "ur

### 5.3 Client 端：官方 SDK 的 ClientSession（`backend/mcp_client/mcp_client.py`）

**关键分工**：

- **Server 端用 FastMCP**：只负责注册工具，装饰器搞定一切
- **Client 端**用官方 `mcp` SDK 的 `ClientSession`【官方 SDK 里内置的是 FastMCP 1.0（2024 年合并进 SDK 的冻结版本），位于 mcp.server.fastmcp，主要定位是服务端装饰器（@mcp.tool() 等），它本身不是一个"客户端框架"；FastMCP v2 独立包 `fastmcp` 另提供 Client，本项目基于官方 SDK 选型】

Client 端我们关心的不是「怎么写协议」，而是**连接类型**：

| 连接类型 | 场景 | 建立会话的方式 |
|---------|------|--------------|
| **stdio** | 内置 Server（本地子进程） | `stdio_client(...)` 返回二元组 (read, write) |
| **streamable_http** | 远程 Server（如 ModelScope 12306） | `streamable_http_client(url)` 返回三元组 |
| **sse** | 远程 Server（旧版 HTTP+SSE，如 ModelScope Fetch） | `sse_client(url)` 返回二元组 (read, write) |


> streamable_http 的握手（initialize → 拿 mcp-session-id → 发 initialized）由 SDK 自动完成。
> 新手最容易踩的坑：**三种连接返回的流数量不一样**——
> `streamable_http_client` 返回**三元组**（多一个 session_id 函数），
> `stdio_client` / `sse_client` 返回**二元组**，解包时不要搞混。


In [14]:
# ============================================================
# Cell 14：看 Client 端源码 —— 三种连接 + 工具格式转换
# ============================================================
from mcp_client import mcp_client as mc

print('=' * 55)
print('  连接工厂 _create_connection —— 按 kind 选择连接类')
print('=' * 55)
print(inspect.getsource(mc._create_connection))

print('=' * 55)
print('  StdioConnection.start（拉起子进程 + 初始化会话）')
print('=' * 55)
print(inspect.getsource(mc.StdioConnection.start))

print('=' * 55)
print('  _to_openai_schema（MCP 工具 -> OpenAI Function Calling Schema）')
print('=' * 55)
print(inspect.getsource(mc._to_openai_schema))

print('[提示] 为什么需要 _to_openai_schema？')
print('   MCP 工具格式和 OpenAI tools 格式不同，注入 LLM 前必须转换')
print('   这就是「MCP 工具也能被 Function Calling 使用」的桥梁')
print()
print('[提示] 三种连接类的差异只在「建立会话的方式」：')
print('   StdioConnection  -> stdio_client（子进程，二元组）')
print('   HttpConnection   -> streamable_http_client（Streamable HTTP，三元组）')
print('   SseConnection    -> sse_client（HTTP+SSE 旧版，二元组）')
print('   连接建立后 list_tools / call_tool 用法完全一致 —— 这就是策略模式')


  连接工厂 _create_connection —— 按 kind 选择连接类
def _create_connection(config: dict) -> BaseConnection:
    """按配置创建对应传输类型的连接（stdio / streamable_http / sse）。

    教学要点——"简单工厂"：
    把"根据 kind 决定实例化哪个类"的逻辑收敛到一个函数里。
    好处：调用方（MCPManager.start / add_server / update_server）不需要写
    if kind == "stdio": ... elif kind == "sse": ... 的长链，
    新增传输类型时也只需改这一个函数。
    """
    kind = config.get("kind", "http")
    if kind == "stdio":
        return StdioConnection(config)
    if kind == "sse":
        return SseConnection(config)
    return HttpConnection(config)

  StdioConnection.start（拉起子进程 + 初始化会话）
    async def start(self) -> None:
        """启动子进程并初始化 MCP 会话。"""
        if self._session is not None:
            return
        self._exit_stack = AsyncExitStack()
        try:
            server_params = StdioServerParameters(
                command=sys.executable,
                args=[str(BUILTIN_SERVER_SCRIPT)],
                env=None,
            )
            read_write = await self._exit_stack

In [15]:
# ============================================================
# Cell 15：演示 —— 真实建立 stdio 会话并调用 web_search
# ============================================================
# 教学要点：MCP 的 stdio 传输需要子进程继承"真实的 stdin/stdout 文件描述符"，
# 而 Jupyter 内核的 stdout 是 ZMQ 虚拟管道（没有 fileno），所以本 Cell 在
# Jupyter 里连接会失败 —— 这是 Jupyter 环境的已知限制，不是代码问题。
# 同样的代码在真实终端运行可以成功（预期结果见下方说明）。
import asyncio
import json
from mcp_client.mcp_client import StdioConnection, _to_openai_schema

async def demo_stdio():
    # 新建一个独立的 stdio 连接（拉起 web_search_server.py 子进程）
    conn = StdioConnection({
        'id': 'demo', 'name': 'demo', 'kind': 'stdio', 'url': '',
        'description': '演示', 'enabled': True, 'builtin': True,
    })
    try:
        await conn.start()
        print('[OK] MCP stdio 会话建立成功')
        print(f'  会话状态: is_running={conn.is_running}')

        tools = await conn.list_tools()
        print(f'  [OK] 发现 {len(tools)} 个工具:')
        for t in tools:
            schema = _to_openai_schema(t)
            print(f'       • {t["name"]} -> '
                  f'{json.dumps(schema, ensure_ascii=False)[:80]}...')

        result = await conn.call_tool('web_search',
                                      {'query': '智能手表 2026', 'num_results': 2})
        print(f'  [OK] 调用 web_search 结果 source = {result.get("source")}')
        print(f'       results 条数 = {len(result.get("results", []))}')
    except Exception as exc:
        if 'fileno' in str(exc):
            # Jupyter 特有失败：说明原因 + 指引到真实终端运行
            print('[NO] 连接失败: fileno')
            print('-' * 55)
            print('原因：Jupyter 内核的 stdout 是 ZMQ 虚拟管道，没有真实')
            print('文件描述符；MCP 的 stdio 传输需要真实的 stdin/stdout，')
            print('因此本 Cell 在 Jupyter 里无法建立 stdio 会话。')
            print('这不是代码问题 —— 同样的代码在真实终端运行可以成功。')
            print('-' * 55)
            print('请在终端运行（配套脚本已生成）：')
            print('    cd smartbrief')
            print('    python demo_stdio.py')
            print('-' * 55)
        else:
            print(f'[NO] 连接失败: {exc}')
    finally:
        await conn.close()
        print('[OK] 会话已关闭')

await demo_stdio()


[NO] 连接失败: fileno
-------------------------------------------------------
原因：Jupyter 内核的 stdout 是 ZMQ 虚拟管道，没有真实
文件描述符；MCP 的 stdio 传输需要真实的 stdin/stdout，
因此本 Cell 在 Jupyter 里无法建立 stdio 会话。
这不是代码问题 —— 同样的代码在真实终端运行可以成功。
-------------------------------------------------------
请在终端运行（配套脚本已生成）：
    cd smartbrief
    python demo_stdio.py
-------------------------------------------------------
[OK] 会话已关闭


In [16]:
# ============================================================
# Cell 16：看 MCPManager —— 多 Server 统一管理 + 当前配置
# ============================================================
print('=' * 55)
print('  MCPManager.start（加载配置并拉起所有启用的 Server）')
print('=' * 55)
print(inspect.getsource(mc.MCPManager.start))

print('=' * 55)
print('  当前 backend/mcp_config.json 配置的 Server')
print('=' * 55)
config = mc._load_config()
for s in config.get('servers', []):
    print(f'  • id={s["id"]}  kind={s["kind"]}  name={s["name"]}  enabled={s["enabled"]}')
    if s.get('url'):
        print(f'    url={s["url"]}')
print()
print('[提示] 12306 是远程 Streamable HTTP Server（ModelScope 托管），')
print('      内置 web_search_server 是本地 stdio Server —— 多类连接并存')

  MCPManager.start（加载配置并拉起所有启用的 Server）
    async def start(self) -> None:
        """加载配置并启动所有启用的 Server（内置 stdio 自动拉起子进程）。"""
        config = _load_config()
        servers = config.get("servers", [])
        # 确保内置 Server 始终存在（不可删除）
        if not any(s.get("id") == BUILTIN_SERVER_ID for s in servers):
            servers.insert(
                0,
                {
                    "id": BUILTIN_SERVER_ID,
                    "name": BUILTIN_SERVER_NAME,
                    "kind": "stdio",
                    "url": "",
                    "description": "内置联网搜索（DuckDuckGo，网络不可用时降级模拟）",
                    "enabled": True,
                    "builtin": True,
                },
            )
            _save_config(servers)
        for cfg in servers:
            self._servers[cfg["id"]] = _create_connection(cfg)
        await self._start_enabled()

  当前 backend/mcp_config.json 配置的 Server
  • id=builtin-web-search  kind=stdio  name=web_search_server  enabled=True
  • id=mcp-2

### 5.4 MCPManager：多 Server 统一管理（容错与约定）

`MCPManager` 是客户端的管理中枢，它的设计里有几个必须记住的约定：

| 约定 | 说明 |
|------|------|
| **Server id 用 uuid** | `mcp-{uuid4().hex[:8]}`，禁止用列表长度递增 id——删除后再添加会复用旧序号，静默覆盖已有 Server |
| **transport 只收 http / sse** | 请求模型层用正则 `^(http|sse)$` 校验，非法值直接返回 422 |
| **内置 Server 只读** | `builtin-web-search` 不可删除/编辑，只可开关；自定义 Server 支持增删改 |
| **8 秒连接超时** | 连接/发现工具用 `asyncio.wait_for(..., timeout=8)` 保护，防止远程 Server 不可达时长时间阻塞 |
| **失败仍保存配置** | 连接失败只记录 `last_error`，配置照常写盘 → 前端提示「已保存，但连接失败」 |
| **可手动刷新工具** | `POST /api/v1/tools/mcp/{id}/refresh` 重新发现工具列表 |

> 💡 设计动机：如果连接失败就抛异常回滚，远程 Server 一挂就永远加不进去，
> 而且「前端报错、后端却没写入」会造成状态不一致。失败保存 + last_error 才是对用户友好的做法。


---
## Part 6：Function Calling 循环（单智能体）
---

### 6.1 需求：完整闭环

现在 LLM、内置工具、MCP 工具都齐了，把它们串成一个循环：

```text
① 组装 messages（含历史）
② 调 LLM（携带 内置 Schema + MCP Schema）
③ 有 tool_calls？ ──否──> 返回最终回答
   └── 是：逐个执行工具 → 把 assistant(含tool_calls) + tool 消息追加回 messages → 回到②
```

### 6.2 两个新手必踩的坑（项目代码里已标注释）

### 6.3 别忘了「角色提示词」

`single_agent.py` 在 `run()` 里把一条 `system` 消息固定放在 messages 最前面，
明确告诉模型：『你是 SmartBrief，一位专业的智能市场调研助手』。
如果没有这条 system prompt，模型容易被 API 的默认身份带偏
（例如回复『我是火车票查询助手』），导致与项目定位不符。

1. **`tool_calls` 不在顶层响应上**：必须从 `resp.choices[0].message.tool_calls` 读
2. **tool 消息前必须回填 assistant 消息**：OpenAI 要求 `role: tool` 之前必须有一条
   包含 `tool_calls` 的 `role: assistant` 消息，否则返回 400 错误

In [17]:
# ============================================================
# Cell 17：看看项目里的最终代码 —— single_agent.py 主循环
# ============================================================
from agents import single_agent

print('=' * 55)
print('  single_agent.py —— run() 主循环')
print('=' * 55)
print(inspect.getsource(single_agent.run))

print('[提示] 观察循环结构：')
print('   • MAX_LOOP = 5：防止模型无限调用工具')
print('   • messages 每轮追加 assistant(带 tool_calls) + tool 消息')
print('   • 工具执行结果记录到 tool_calls 列表，返回给前端展示')

  single_agent.py —— run() 主循环
async def run(user_input: str, history: list | None = None) -> dict:
    """单智能体对话入口（Function Calling 主循环）。

    :param user_input: 用户提问（自然语言，可触发工具调用）
    :param history: 历史消息（OpenAI 格式），可选
    :return: {"reply": 最终回答, "tool_calls": 本次工具调用记录列表}

    教学要点——整个循环在做什么：
    1) 组装 messages（system 人设 + 历史 + 当前提问）
    2) 调用 LLM；模型可能有两种反应：
       - 直接给最终回答（message.tool_calls 为空）→ 返回，循环结束
       - 要求调用工具（message.tool_calls 非空）→ 我们执行工具，把结果
         以 tool 消息回填，**再带着新消息重新问一次模型**（模型会基于
         工具结果继续回答，或继续要求调下一个工具）
    3) 因为可能连环调用（比如先搜索再计算），所以要 for 循环；
       但又必须设 MAX_LOOP 上限，防止模型陷入无限调用（死循环烧钱）。
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    # 过滤掉历史中的 system 消息，避免重复注入或用户篡改
    for msg in history or []:
        if msg.get("role") == "system":
            continue
        messages.append(msg)
    # 在 user 消息里再次强调 SmartBrief 身份，防止模型被 API 默认身份覆盖
    wrapped_input = f"[Remember: you are SmartBrief, a market research assistant] {user_input}"
    

In [18]:
# ============================================================
# Cell 18：手动模拟「消息回填」的协议细节（不调用模型）
# ============================================================
import json

# 模拟：模型第一轮返回了一个 tool_calls
messages = [{'role': 'user', 'content': '现在几点了？'}]

# ① 回填 assistant 消息（必须包含 tool_calls！）
messages.append({
    'role': 'assistant',
    'content': None,
    'tool_calls': [
        {'id': 'call_demo_1', 'type': 'function',
         'function': {'name': 'current_time', 'arguments': '{}'}}
    ],
})

# ② 执行工具（真实调用 current_time）
import asyncio
result = await builtin_tools.current_time()

# ③ 追加 tool 消息（必须带 tool_call_id 对应上面的 id）
messages.append({
    'role': 'tool',
    'tool_call_id': 'call_demo_1',
    'content': json.dumps(result, ensure_ascii=False),
})

print('=' * 55)
print('  一轮工具调用后的 messages 结构')
print('=' * 55)
for m in messages:
    print(f'  role={m["role"]:<10} keys={list(m.keys())}')
    if m['role'] == 'tool':
        print(f'    tool_call_id={m["tool_call_id"]}  content={m["content"]}')
    if 'tool_calls' in m:
        print(f'    tool_calls={[tc["function"]["name"] for tc in m["tool_calls"]]}')

  一轮工具调用后的 messages 结构
  role=user       keys=['role', 'content']
  role=assistant  keys=['role', 'content', 'tool_calls']
    tool_calls=['current_time']
  role=tool       keys=['role', 'tool_call_id', 'content']
    tool_call_id=call_demo_1  content="2026-09-01 00:03:34"


In [19]:
# ============================================================
# Cell 19：真实演示 —— 完整单智能体对话（需要 .env key + 网络）
# ============================================================
from agents.single_agent import run as agent_run

async def demo_agent():
    try:
        # 提问一：触发内置工具 current_time
        print('=' * 55)
        print('  问题：现在几点了？')
        print('=' * 55)
        r1 = await agent_run('现在几点了？')
        for tc in r1['tool_calls']:
            print(f'  [工具] {tc["name"]}({tc["arguments"]}) success={tc["success"]}')
        print(f'  [回答] {r1["reply"][:100]}')

        # 提问二：触发内置工具 calculator
        print()
        print('=' * 55)
        print('  问题：计算 12*34+56 等于多少？')
        print('=' * 55)
        r2 = await agent_run('计算 12*34+56 等于多少？')
        for tc in r2['tool_calls']:
            print(f'  [工具] {tc["name"]}({tc["arguments"]}) -> {tc["result"]}')
        print(f'  [回答] {r2["reply"][:100]}')

        # 提问三：触发 MCP 工具 web_search（网络不可达时降级 mock）
        print()
        print('=' * 55)
        print('  问题：搜索 便携储能 2026 市场')
        print('=' * 55)
        r3 = await agent_run('搜索 便携储能 2026 市场')
        for tc in r3['tool_calls']:
            print(f'  [工具] {tc["name"]} success={tc["success"]}')
            if isinstance(tc.get('result'), dict) and 'source' in tc['result']:
                print(f'         数据来源: {tc["result"]["source"]}')
        print(f'  [回答] {r3["reply"][:100]}')
    except Exception as exc:
        print(f'[NO] 对话失败: {exc}')
        print('     原因通常是未配置 OPENAI_API_KEY 或网络不可达')

await demo_agent()

  问题：现在几点了？


[09/01/26 00:03:34] INFO     HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 ]8;id=921149;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921150;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             200 OK"                                                                               

[09/01/26 00:03:36] INFO     HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 ]8;id=921155;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921156;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             200 OK"                                                                               

  [工具] current_time({}) success=True
  [回答] 现在是 **2026年9月1日 00:03:36**（服务器时间）。

我是 **SmartBrief**，一名专业的市场研究助理。我的职责是帮助您调研产品或行业，并生成结构化的市场简报。请问您今天想

  问题：计算 12*34+56 等于多少？


[09/01/26 00:03:38] INFO     HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 ]8;id=921161;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921162;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             200 OK"                                                                               

[09/01/26 00:03:40] INFO     HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 ]8;id=921167;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921168;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             200 OK"                                                                               

  [工具] calculator({'expression': '12*34+56'}) -> 464
  [回答] 12 × 34 + 56 = **464**

（计算过程：12 × 34 = 408，408 + 56 = 464）

顺便自我介绍一下，我是 **SmartBrief**，一名专业的市场调研助手。

  问题：搜索 便携储能 2026 市场


[09/01/26 00:03:42] INFO     HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 ]8;id=921173;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921174;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             200 OK"                                                                               

[09/01/26 00:03:45] INFO     HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 ]8;id=921179;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=921180;file://c:\Users\13691\.conda\envs\pg-env\lib\site-packages\httpx\_client.py#1740\1740]8;;\
                             200 OK"                                                                               

  [工具] current_time success=True
  [回答] 您好，我是 SmartBrief，您的市场研究助手。以下是我基于行业知识为您整理的 **2026 年便携储能（Portable Power Station）市场简报**。

> ⚠️ 说明：当前会话中


---
## Part 7：把一切串起来（API 与前端）
---

### 7.1 后端入口：`backend/main.py`

FastAPI 用 `lifespan` 管理启动/关闭钩子：

- 启动时：初始化内置工具注册表 → `await mcp_manager.start()` 拉起全部启用的 MCP Server
- 关闭时：`await mcp_manager.close()` 回收子进程与 HTTP 连接

In [20]:
# ============================================================
# Cell 20：看 main.py + 打印 API 路由表
# ============================================================
import main as backend_main

print('=' * 55)
print('  main.py —— lifespan')
print('=' * 55)
print(inspect.getsource(backend_main.lifespan))

print('=' * 55)
print('  全部 API 路由')
print('=' * 55)
from api.routes import router
for route in router.routes:
    methods = ','.join(sorted(route.methods - {'HEAD', 'OPTIONS'}))
    print(f'  {methods:<7} {route.path}')

print()
print('[提示] 前端通过 Vite 代理把 /api 转发到 8002 端口，'
      '所以浏览器里请求的是 /api/v1/chat，实际落到后端')

  main.py —— lifespan
@asynccontextmanager
async def lifespan(app: FastAPI):
    """应用生命周期钩子：启动时初始化三大组件，关闭时回收。

    初始化顺序有讲究（后面的依赖前面的）：
    1. builtin_tools.init()：先建工具注册表（应用 tool_config.json 开关）
    2. skills_manager.init()：技能池加载（内置技能内部会引用 mcp_manager）
    3. await mcp_manager.start()：最后拉起 MCP Server（内置 stdio 自动启动子进程）
    """
    # 初始化内置工具注册表（应用持久化的开关状态）
    builtin_tools.init()
    # 初始化技能池：注册内置技能 + 加载 skills_pool/ 下的自定义技能
    skills_manager.init()
    # 加载 MCP Server 配置并启动所有启用的 Server（内置 stdio 自动拉起子进程）
    await mcp_manager.start()
    yield
    # 关闭时回收子进程 / HTTP 连接（避免僵尸进程占用端口）
    await mcp_manager.close()

  全部 API 路由
  GET     /api/v1/health
  POST    /api/v1/chat
  GET     /api/v1/tools
  GET     /api/v1/tools/builtin
  PUT     /api/v1/tools/builtin/{name}
  GET     /api/v1/tools/mcp
  POST    /api/v1/tools/mcp
  PUT     /api/v1/tools/mcp/{server_id}
  DELETE  /api/v1/tools/mcp/{server_id}
  PUT     /api/v1/tools/mcp/{server_id}/enabled
  POST    /api/v1/tools/mcp/{server_id}/re

### 7.2 前端：AI 对话页 + 工具监控页

前端只有两个页面：

- **AI 对话**（`frontend/src/views/Chat.vue`）：聊天窗口，工具调用以卡片展示（名称/参数/结果/成败）
- **工具监控**（`frontend/src/views/ToolMonitor.vue`）：双 Tab 管理内置工具（开关）与 MCP 工具（开关/增删改/刷新）

多轮对话的上下文由**前端维护**：每次请求把历史 `history` 一起回传，后端拼进 messages。

> 🔌 若想在上面的 Cell 19 完整演示 `web_search` 的 MCP 调用，请先启动后端
> （`cd backend && uvicorn main:app --reload --port 8002`），或在运行前执行一次
> `await mc.mcp_manager.start()`（notebook 中直接用顶层 `await`，不要用 `asyncio.run()`）。

In [21]:
# ============================================================
# Cell 21：瞄一眼前端 API 封装 chat.ts
# ============================================================
chat_ts = Path('frontend/src/api/chat.ts')
if chat_ts.exists():
    print('=' * 55)
    print('  frontend/src/api/chat.ts')
    print('=' * 55)
    print(chat_ts.read_text(encoding='utf-8')[:1200])
    print()
    print('[提示] sendChat 把 history 一起回传 —— 这就是多轮对话的上下文来源')
else:
    print('[提示] 前端文件不在，请确认工作目录是 smartbrief/')

  frontend/src/api/chat.ts
import axios from 'axios'

/** 工具调用记录 */
export interface ToolCallRecord {
  name: string
  arguments: Record<string, unknown>
  result: unknown
  success: boolean
}

/** 单智能体对话响应 */
export interface ChatResult {
  reply: string
  tool_calls: ToolCallRecord[]
}

const http = axios.create({
  baseURL: '/api/v1',
  timeout: 60000 // 对话涉及多轮工具调用，放宽超时
})

/** 发送消息给单智能体，触发工具调用（提示词触发） */
export async function sendChat(message: string, history: unknown[] = []): Promise<ChatResult> {
  const { data } = await http.post('/chat', { message, history })
  return data as ChatResult
}


[提示] sendChat 把 history 一起回传 —— 这就是多轮对话的上下文来源


---
## Part 8：课程总结
---

### 这节课我们经历了什么？

不是「学会了几个类」，而是经历了一个真实开发者的思考过程：

| 演进阶段 | 遇到的问题 | 解决方案 |
|---------|-----------|---------|
| v0 | 密钥硬编码、配置写死 | 配置化 + `.env` 读取 + 懒加载单例 |
| v1 | 用 if/else 猜意图，规则写不完 | 让「模型猜意图」——Function Calling |
| v2 | 函数散落、无法统一管理 | 工具注册表（函数 + Schema + 开关） |
| v3 | eval 执行任意代码，致命漏洞 | AST 白名单安全求值 |
| v4 | 每次都要自己造轮子（搜索） | MCP 协议：FastMCP Server + 官方 SDK Client |
| v5 | 工具调用不成闭环 | Function Calling 循环（≤5 轮） |

### 关键设计原则

1. **配置与代码分离**：密钥一律 `.env`，代码零硬编码
2. **契约优先**：JSON Schema 是模型与函数的接口契约
3. **注册表模式**：工具集中注册、统一路由、可开关、可持久化
4. **白名单安全**：绝不 eval 用户输入，AST 逐节点校验
5. **协议标准化**：能力用 MCP 暴露，消费方与实现方解耦
6. **分层清晰**：LLM 封装 / 工具 / MCP / Agent 各司其职

### 下节课预告

**第 23 节：技能（Skill）抽象与管理——技能池**

- 在工具调用之上抽象技能层，实现技能的增删改查（CRUD）
- 内置技能与自定义技能分开管理，支持 .zip 导入与 skill.md 创建

---
## 课后练习
---

### 练习一：新增一个内置工具

基于注册表模式，给系统新增一个「字符串反转」工具 `reverse_text`：

1. 参考 `calculator` 写一个 Schema（参数：`text: string`）
2. 写执行函数并注册进 `init()`
3. 思考：需要修改哪些代码？新增哪些代码？（提示：零 if/else）

### 练习二：手动走一遍协议

用 Cell 18 的方式，模拟一轮「调用 web_search」的完整消息回填：

1. 构造 assistant 消息（带 tool_calls）
2. 调用 `mcp_manager.call_tool('web_search', ...)`
3. 追加 tool 消息，观察 messages 结构

### 练习三：安全求值加固

试着让 `_safe_eval` 支持更多合法运算：

1. 支持整数除法 `//`（提示：`ast.FloorDiv`）
2. 支持逻辑与 `and` / `or`（提示：`ast.And` / `ast.Or`）
3. 改完用恶意表达式测试，确认仍然拒绝函数调用

In [22]:
# ============================================================
# Cell 22：练习参考 —— 练习三：给 _safe_eval 增加 // 支持
# ============================================================
import ast
from tools.builtin_tools import _BINARY_OPS

# 方案：扩展白名单字典，映射 ast.FloorDiv（整数除法）
extended_ops = dict(_BINARY_OPS)
extended_ops[ast.FloorDiv] = lambda a, b: a // b

def safe_eval_ext(node):
    if isinstance(node, ast.Expression):
        return safe_eval_ext(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in extended_ops:
        return extended_ops[type(node.op)](
            safe_eval_ext(node.left), safe_eval_ext(node.right))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -safe_eval_ext(node.operand)
    raise ValueError('仅支持数字、四则运算与括号')

print('=' * 55)
print('  扩展安全求值：新增 // 运算符')
print('=' * 55)
for expr in ['7 // 2', '10 / 4', '(7 + 1) // 2']:
    print(f'  {expr:>12} = {safe_eval_ext(ast.parse(expr, mode="eval"))}')

# 恶意表达式仍然被拒绝
try:
    safe_eval_ext(ast.parse("__import__('os').getcwd()", mode='eval'))
    print('  [NO] 恶意输入竟通过了！')
except (ValueError, SyntaxError) as exc:
    print(f'  [OK] 恶意输入仍被拒绝: {exc}')

print()
print('[提示] 同理可加幂 ** 与取余 %（它们已在项目白名单里）')
print('      扩展的哲学：白名单里加「运算符节点类型」，而不是放开 eval')

  扩展安全求值：新增 // 运算符
        7 // 2 = 3
        10 / 4 = 2.5
  (7 + 1) // 2 = 4
  [OK] 恶意输入仍被拒绝: 仅支持数字、四则运算与括号

[提示] 同理可加幂 ** 与取余 %（它们已在项目白名单里）
      扩展的哲学：白名单里加「运算符节点类型」，而不是放开 eval


---

## 📚 参考资料

- 本节需求文档：《多智能体协作开发PRD（拆分）.md》第 22 节（项目根目录）
- LLM 封装：`backend/llm/client.py`
- 内置工具：`backend/tools/builtin_tools.py`
- MCP Server：`mcp_servers/web_search_server.py`
- MCP 客户端：`backend/mcp_client/mcp_client.py`
- 单智能体：`backend/agents/single_agent.py`

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道代码长什么样，更知道代码为什么这么长这样。